In [7]:
import time
from datetime import datetime
from io import BytesIO
from pathlib import Path
import requests
from PIL import Image

from tile_utils import latlon_to_tile, latlon_to_pixel, draw_marker


# Configuración por proveedor: URL template, orden de x/y, zoom máximo típico,
# y pausa recomendada entre requests (cada servicio tiene su propia política de uso).
PROVIDERS = {
    "opentopomap": {
        "url_template": "https://tile.opentopomap.org/{z}/{x}/{y}.png",
        "max_zoom": 17,
        "sleep": 0.5,
    },
    "esri": {
        # OJO: ESRI usa el orden {z}/{y}/{x}, al revés del estándar {z}/{x}/{y}
        "url_template": "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        "max_zoom": 17,
        "sleep": 0.2,
    },
}


def get_point_map(lat, lon, provider="esri", zoom=None, radius=2,
                   out_prefix=None, output_dir="../img/maps", show_marker=True):
    """
    lat, lon: coordenadas del punto central
    provider: "esri" (foto satelital real) u "opentopomap" (mapa topográfico/altitud)
    zoom: si no se especifica, usa el max_zoom típico del proveedor
    radius: cuántos tiles agregar alrededor del centro en cada dirección
    out_prefix: nombre base del archivo; si no se especifica, usa el nombre del proveedor
    output_dir: carpeta donde se guarda la imagen (se crea sola si no existe)
    show_marker: si True, dibuja un punto rojo en la ubicación exacta
    """
    if provider not in PROVIDERS:
        raise ValueError(f"Proveedor '{provider}' no reconocido. Opciones: {list(PROVIDERS.keys())}")

    config = PROVIDERS[provider]
    zoom = zoom or config["max_zoom"]
    out_prefix = out_prefix or f"point_{provider}"

    timestamp = datetime.now().strftime("%y%m%d%H%M%S")
    filename = f"{out_prefix}-v{timestamp}.png"
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    out_path = output_path / filename

    x_center, y_center = latlon_to_tile(lat, lon, zoom)
    x_min, x_max = x_center - radius, x_center + radius
    y_min, y_max = y_center - radius, y_center + radius

    tile_size = 256
    width = (x_max - x_min + 1) * tile_size
    height = (y_max - y_min + 1) * tile_size
    canvas = Image.new("RGB", (width, height))

    headers = {"User-Agent": "agri_land_suitability_pipeline (tu_email_real@dominio.com)"}

    for x in range(x_min, x_max + 1):
        for y in range(y_min, y_max + 1):
            url = config["url_template"].format(z=zoom, x=x, y=y)
            resp = requests.get(url, headers=headers, timeout=10)

            if resp.status_code != 200:
                print(f"Tile {x},{y} falló con status {resp.status_code}")
                time.sleep(config["sleep"])
                continue

            try:
                tile_img = Image.open(BytesIO(resp.content))
                canvas.paste(tile_img, ((x - x_min) * tile_size, (y - y_min) * tile_size))
            except Exception as e:
                print(f"Tile {x},{y} falló: {e}")

            time.sleep(config["sleep"])

    if show_marker:
        px_world, py_world = latlon_to_pixel(lat, lon, zoom, tile_size)
        px_canvas = px_world - x_min * tile_size
        py_canvas = py_world - y_min * tile_size
        draw_marker(canvas, px_canvas, py_canvas)

    canvas.save(out_path)
    print(f"Imagen guardada en {out_path} ({width}x{height}px)")
    return out_path


if __name__ == "__main__":
    # Foto satelital real
    get_point_map(7.3297, -73.1867, provider="esri", radius=2)

    # Mapa de altitud/relieve (si lo necesitas de nuevo más adelante)
    # get_point_map(7.3297, -73.1867, provider="opentopomap", radius=2)

Imagen guardada en ../img/maps/point_esri-v260804194710.png (1280x1280px)
